# Entrenar YOLO propio — Cirbonero vs Milagres

Entrena un YOLO11s con **nuestros** fotogramas del partido (210 imágenes, 168 train / 42 val, clases: `0=football`, `1=player`).

**Requisitos:** GPU en Colab (Entorno > Cambiar tipo de entorno > GPU T4). El dataset ya está etiquetado:
- Etiquetas + `data.yaml` en el repo GitHub (`dataset/`).
- Imágenes en Drive, carpeta **Git Hub** (`dataset_cirbonero_milagres_images.zip`).

Al final el modelo queda en Drive (`Mi unidad/Git Hub/modelo_propio/best.pt`) para usarlo en `match.py` / `video_track.py`.

In [ ]:
# 1. Instalar ultralytics
!pip install -q ultralytics

In [ ]:
# 2. Montar Drive y preparar el dataset
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

BASE = '/content/match_view'
os.makedirs(BASE, exist_ok=True)

# Imágenes: zip en Drive (carpeta "Git Hub")
ZIP = '/content/drive/MyDrive/Git Hub/dataset_cirbonero_milagres_images.zip'
with zipfile.ZipFile(ZIP) as z:
    z.extractall(f'{BASE}/dataset')
print('imágenes:', len(os.listdir(f'{BASE}/dataset/images/train')) + len(os.listdir(f'{BASE}/dataset/images/val')))

In [ ]:
# 3. Bajar etiquetas + data.yaml del repo
!git clone -q https://github.com/Josen-IG/Match_view.git /content/repo_tmp

import shutil, os
# etiquetas (labels) del repo -> dataset/
for split in ('train', 'val'):
    src = f'/content/repo_tmp/dataset/labels/{split}'
    dst = f'{BASE}/dataset/labels/{split}'
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
# data.yaml del repo (apunta a images/train y images/val)
shutil.copy('/content/repo_tmp/dataset/data.yaml', f'{BASE}/dataset/data.yaml')
print(open(f'{BASE}/dataset/data.yaml').read())

In [ ]:
# 4. Entrenar
# yolo11s.pt = base (misma arquitectura YOLO, pesos propios tras entrenar).
# imgsz 960 = el mismo que usa el análisis; batch 16 en T4 (si da OOM, baja a 8).
# epochs 50 con paciencia 10: el dataset es pequeño (168 train), no conviene mucho más.
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
model.train(
    data=f'{BASE}/dataset/data.yaml',
    epochs=50,
    imgsz=960,
    batch=16,
    patience=10,
    project=f'{BASE}/runs',
    name='cirbonero_milagres',
    device=0,
)

In [ ]:
# 5. Validar y ver el resultado
from ultralytics import YOLO
best = YOLO(f'{BASE}/runs/cirbonero_milagres/weights/best.pt')
metrics = best.val(data=f'{BASE}/dataset/data.yaml', imgsz=960)
print('mAP50:', round(metrics.box.map50, 3))
print('mAP50-95:', round(metrics.box.map, 3))

In [ ]:
# 6. Guardar el modelo en Drive (para usarlo en match.py / video_track.py)
import shutil, os
OUT = '/content/drive/MyDrive/Git Hub/modelo_propio'
os.makedirs(OUT, exist_ok=True)
shutil.copy(f'{BASE}/runs/cirbonero_milagres/weights/best.pt', f'{OUT}/best.pt')
shutil.copy(f'{BASE}/runs/cirbonero_milagres/weights/last.pt', f'{OUT}/last.pt')
print('guardado en:', OUT)
print('tamaño best.pt (MB):', round(os.path.getsize(f'{OUT}/best.pt') / 1e6, 1))

## Usarlo en el análisis

En `match.py` / `video_track.py`, cambia el modelo a la ruta de `best.pt` (descargado de Drive) y vuelve a analizar. Si el balón mejora mucho, se puede afinar con más épocas o más fotogramas etiquetados.

**Nota:** el modelo entrena solo `football` y `player`. El árbitro (clase 2) está pendiente de etiquetar a mano; hasta entonces el árbitro se clasifica por color, como ahora.